In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import plotly.express as px

## SIRS Model with Human Behaviour
The model studied below is the following:
$$
\begin {cases}
\dot{S} = (\mu + \theta) - (\mu + \theta) S - \beta(M)SI - \theta I \\
\dot{I} = I( \beta(M)S - (\mu + \gamma) ) \\
\dot{M} = a(I - M)
\end{cases}
$$
, where we arbitrary chose:
$$
\beta(M) = \frac{\beta_{0}}{1 + cM},  \qquad
M(t) = \int_{0}^{t}W(\tau)I(t -\tau)d\tau, \qquad
W(\tau) = a e^{-a \tau}
$$

In [ ]:
def beta_func(beta_0, c):
    def beta_function(x):
        return beta_0 / (1 + c * x)

    return beta_function

In [ ]:
def SIRS_hb(t, X, beta, gamma, mu, theta, a):
    S, I, M = X

    dS = (mu + theta) - (mu + theta) * S - beta(M) * I * S - theta * I
    dI = I * (beta(M) * S - (mu + gamma))
    dM = a * (I - M)

    return [dS, dI, dM]

In [ ]:
# parameters

# for now, same as above
# t = 1 month
# gamma = 0.0167 * 30  # Recovery rate
# mu = 0.000083 * 30  # Birth/death rate
# theta = 0.00027 * 30  # Loss of immunity rate
# t from the study
gamma = 0.5
mu = 0.03 / 12
theta = 0.05 / 12

a = 0.042     # (Characteristic memory length)^-1 (24 months)
# beta_0 = 0.5027
# beta_0 = 1.000*(mu+gamma)
# beta_0 = 1.9*(mu+gamma)
# beta_0 = 5*(mu+gamma)
# beta_0 = 0.8*(mu+gamma)
beta_0 = 1.*(mu+gamma)
# beta_0 = 0.9*(mu+gamma)
c = 2
beta = beta_func(beta_0, c)

# Initial conditions
S0 = 0.999  # Initial suspicious population
I0 = 0.001  # Initial infected population
M0 = 0      # Initial memory rate
X0 = [S0, I0, M0]

# t = 1 month
t_span = (0, 1200)

In [ ]:
# Solve the system of ODEs
solution = solve_ivp(SIRS_hb, t_span, X0, args=(beta, gamma, mu, theta, a), dense_output=True)

# Time points for which to get the solution
t = np.linspace(t_span[0], t_span[1], 100000)
sol = solution.sol(t)

In [ ]:
# Calculating some specifics of the model

# Some specifics of the model
mean_R_0 = float(np.mean([beta(M) / (mu + gamma) for M in sol[2]]))
beta_e = beta(sol[2][-1])
R_0_ee = beta_e / (mu + gamma)

# Endemic equilibrium
S_e = float(1/R_0_ee)
I_e = float((1 - 1/R_0_ee) * (mu + theta) / (mu + gamma + theta))
R_e = float((1 - 1/R_0_ee) * gamma / (mu + gamma + theta))

In [ ]:
# print parameters
print(f"Parameters: \nGamma: {gamma}\t\tMu: {mu}\t\tTheta: {round(theta, 5)}\t\tR0 at equilibrium: {round(R_0_ee, 5)}\n")
print(f"Specifics: \nEndemic Equilibrium: {round(S_e, 10), round(I_e, 5), round(R_e, 5)}")

In [ ]:
fig = px.line(x=t, y=[0] * len(sol[0]), title='SIRS - Human Behaviour', labels={'x': 'Time', 'y': 'Population'}, color_discrete_sequence=['rgba(0, 0, 0, 0)'])
#
fig.add_scatter(x=t, y=sol[0], mode='lines', name='Suspicious', line=dict(color='blue'))
fig.add_scatter(x=t, y=sol[1], mode='lines', name='Infected', line=dict(color='red'))
fig.add_scatter(x=t, y=1 - sol[0] - sol[1], mode='lines', name='Recovered', line=dict(color='green'))
#
fig.show()

In [ ]:
fig = px.line(x=t[0:40000], y=[0] * len(sol[1][:40000]), title='Human Behaviour and Infectious - I(t) and M(t)',
                  labels={'x': 'Time', 'y': 'Population'}, color_discrete_sequence=['rgba(0, 0, 0, 0)'])
fig.add_scatter(x=t[0:40000], y=sol[1][:40000], mode='lines', name='I(t)', line=dict(color='red'))
fig.add_scatter(x=t[0:40000], y=sol[2][:40000], mode='lines', name='M(t)', line=dict(color='magenta'))

fig.update_layout(
    yaxis=dict(
        tickmode = 'linear',
        dtick = 0.02
    )
)

fig.show()

In [ ]:
betas = [beta(i) for i in sol[2]]
one_betas = [beta_0 - i for i in betas]

In [ ]:
fig = px.line(x=t[:50000], y=[0]*50000, title=f'Human Behaviour   -   M(t) and β₀-β(M)', color_discrete_sequence=['rgba(0,0,0,0)'], labels={'x': 'Time', 'y': 'Population'})
fig.add_scatter(x=t[:50000], y=sol[2][:50000], mode='lines', name='M(t)')
# fig_betas.add_scatter(x=t[:50000], y=sol[1][:50000], mode='lines', name='I(t)')
fig.add_scatter(x=t[:50000], y=one_betas[:50000], mode='lines', name='β₀ - β(M)')

fig.update_layout(
    yaxis = dict(
        tickmode = 'linear',
        dtick = 0.04
    )
)

fig.show()

In [ ]:
# Routh-Hurwitz equation for Hopf's bifurcations given 'a' as the free parameter
# equation is in the following form:
# (c3 + c2)a^2 + (c3^2 + c3c2 + c1 - c0)a + c3c1 = 0

c3 = (mu + gamma) + beta_e*I_e
c2 = -c*beta_0 / (1 + c*I_e)**2 * I_e * S_e
c1 = beta_e * I_e * (mu + gamma + theta)
c0 = - (-c2*(mu + theta) - c1)

In [ ]:
def routh(c3, c2, c1, c0):
    def routh_inside(a):
        return (c3 + c2)*a**2 + (c3**2 + c3*c2 + c1 - c0)*a + c3*c1
    return routh_inside

In [ ]:
a_s = [i for i in np.arange(0, 0.1, 0.00001)]

a_function = routh(c3, c2, c1, c0)
results = [a_function(a) for a in a_s]

In [ ]:
fig = px.line(x=a_s, y=results, title=f'Routh-Hurwitz Condition for β₀={round(beta_0, 3)}, Mean R₀={round(R_0_ee, 3)}', labels={'x':'a', 'y':'RH(a)'})
fig.show()

In [ ]:
print(f"Routh-Hurwitz(0): {round(a_function(0), 10)}\n"
      f"Vertex: {round(-(c3**2 + c3*c2 + c1 - c0) / (2*(c3 + c2)), 5)}")

In [ ]:
d3 = beta_e*I_e + mu + theta
d2 = beta_e * I_e * (mu + theta)
d1 = - beta_e * I_e *gamma

In [ ]:
d3**4 - 4*d3*(d3*d2-d1), d3, d2, d3**2, d3*d2

In [ ]:
def routh_b(c3, c2, c1):
    def routh_inside(a):
        return c3 * (a ** 2) + (c3 ** 2)* a + c3 * c2 - c1
    return routh_inside


a_s = [i for i in np.arange(-1, 1, 0.001)]

a_function = routh_b(d3, d2, d1)
results = [a_function(a) for a in a_s]
fig = px.line(x=a_s, y=results,
              title=f'Routh-Hurwitz Condition for β₀={round(beta_e, 9)}, Mean R₀={round(R_0_ee, 9)}',
              labels={'x': 'a', 'y': 'RH(a)'})
fig.show()

## SIRS with Human Behaviour and Double Memory
The model here is similar to the previous one, but instead we choose:
$$
W(t) = a \tau e^{-a \tau}
$$
so that the resulting equations for the memory are:
$$
\begin{cases}
\dot{M_1} = a_1 (I - M_1) \\
\dot{M_2} = a_2 (M_1 - M_2)
\end{cases}
$$
I.e. this is the case in which people have two different *layers* of memory, one that is very close to the present and one other very aged.

At the same time we had to change the function for $\beta$, so that it reflects the changing:
$$
\beta(M) = \beta(M_1, M_2) = \frac{\beta_0}{(1 + c_1 M_1)(1 + c_2 M_2)}
$$

In [ ]:
def SIRS_double (t, X, beta, gamma, mu, theta, a1, a2):
    S, I, M1, M2 = X

    dS = (mu + theta) - S*(mu + theta) - beta(M1, M2)*I*S - theta*I
    dI = I*( beta(M1, M2)*S - (mu + gamma) )
    dM1 = a1 * (I - M1)
    dM2 = a2 * (M1 - M2)

    return [dS, dI, dM1, dM2]

In [ ]:
def beta_double (beta0, d1, d2):
    def beta_double_inside(M1, M2):
        return beta0 / (1+d1*M1) / (1+d2*M2)
    return beta_double_inside

In [ ]:
# parameters

# for now, same as above
gamma = 0.5
mu = 0.03 / 12
theta = 0.05 / 12

a1 = 0.000095                  # Short term memory  characteristic time        3 months
a2 = 0.000099                 # Long term memory characteristic time          5 years
beta_0 = 1.9*(mu+gamma)

d1 = 2                      # Short term memory weight
d2 = 10                      # Long term memory weight
beta = beta_double(beta_0, d1, d2)

# Initial conditions
S0 = 0.999                  # Initial suspicious population
I0 = 0.001                  # Initial infected population
M10 = 0                     # Initial short memory rate
M20 = 0.0                   # Initial long term memory rate
X0 = [S0, I0, M10, M20]

# t = 100 month
t_span = (0, 12000)

In [ ]:
# Solve the system of ODEs
solution = solve_ivp(SIRS_double, t_span, X0, args=(beta, gamma, mu, theta, a1, a2), dense_output=True)

# Time points for which to get the solution
t = np.linspace(t_span[0], t_span[1], 100000)
sol = solution.sol(t)

In [ ]:
# Calculating some specifics of the model

# Some specifics of the model
mean_m1 = np.mean(sol[2])
mean_m2 = np.mean(sol[3])
mean_R_0 = beta(mean_m1, mean_m2) / (mu + gamma)
beta_e = beta(sol[2][-1], sol[3][-1])
R_0_ee = beta_e / (mu + gamma)

# Endemic equilibrium
S_e = float(1 / R_0_ee)
I_e = float((1 - 1 / R_0_ee) * (mu + theta) / (mu + gamma + theta))
R_e = float((1 - 1 / R_0_ee) * gamma / (mu + gamma + theta))
# print parameters
print(
    f"Parameters: \nGamma: {gamma}\t\tMu: {mu}\t\tTheta: {round(theta, 5)}\t\tR0 at equilibrium: {round(R_0_ee, 5)}\n")
print(f"Specifics: \nEndemic Equilibrium: {round(S_e, 10), round(I_e, 5), round(R_e, 5)}")

In [ ]:
fig = px.line(x=t, y=[0] * len(sol[0]), title='SIRS - Human Behaviour', labels={'x': 'Time', 'y': 'Population'}, color_discrete_sequence=['rgba(0, 0, 0, 0)'])
#
fig.add_scatter(x=t, y=sol[0], mode='lines', name='Suspicious', line=dict(color='blue'))
fig.add_scatter(x=t, y=sol[1], mode='lines', name='Infected', line=dict(color='red'))
fig.add_scatter(x=t, y=1 - sol[0] - sol[1], mode='lines', name='Recovered', line=dict(color='green'))
#
fig.show()

In [ ]:
fig = px.scatter(x = t[:40000], y=[0]*len(sol[2][:40000]), title='I(t) and the memories M1(t) and M2(t)', labels={'x':'Months', 'y':'Value'}, color_discrete_sequence=['rgba(0, 0, 0, 0)'])
fig.add_scatter(x=t[:40000], y=sol[1][:40000], name='I(t)', line=dict(color='red'))
fig.add_scatter(x=t[:40000], y=sol[2][:40000], name='M1(t)', line=dict(color='blue'))
fig.add_scatter(x=t[:40000], y=sol[3][:40000], name='M2(t)', line=dict(color='rgb(0, 125, 255)'))

fig.show()

In [ ]:
betas_double = [beta(m1, m2) for m1, m2 in zip(sol[2], sol[3])]
one_betas_double = [beta_0 - i for i in betas_double]

In [ ]:
fig = px.scatter(x=t[:30000], y=[0]*len(sol[2][:30000]), title='β₀-β(t)', color_discrete_sequence=['rgba(0, 0, 0, 0)'])
fig.add_scatter(x=t[:30000], y=sol[2][:30000], name='M1', line=dict(color='blue'))
fig.add_scatter(x=t[:30000], y=sol[3][:30000], name='M2', line=dict(color='rgb(0, 125, 255)'))
fig.add_scatter(x=t[:30000], y=one_betas_double[:30000], name='β₀-β(t)', line=dict(color='aquamarine'))

fig.show()

### Routh-Hurwitz Function plot

$$q_1^2 - q_1q_2q_3 + q_0q_3^2 = 0$$

In [ ]:
def beta_derivative(beta_0, d1, d2):
    def beta_derivative_inside(M1, M2, idx):
        res = - beta_0 * d2 /( (1+d1*M1) * (1 + d2*M2)**2) if idx == 1 else - beta_0*d1 / ((1+d2*M2) * (1 + d1*M1)**2)
        return res
    return beta_derivative_inside

In [ ]:
beta_d = beta_derivative(beta_0, d1, 2)
beta_ds_1 = [beta_d(M1, M2, 0) for M1, M2 in zip(sol[2], sol[3])]
beta_ds_2 = [beta_d(M1, M2, 1) for M1, M2 in zip(sol[2], sol[3])]

db1_e = beta_ds_1[-1]
db2_e = beta_ds_2[-1]

In [ ]:
c3 = beta_e*I_e+ mu + theta
c2 = beta_e*I_e*(mu + gamma + theta)
c1 = -db1_e*I_e*S_e
c0 = -db2_e*I_e*S_e

In [ ]:
q3 = a1 + a2 + c3
q2 = a1*a2 + a1*(c1+c3) + a2*c3 + c2
q1 = a1*a2*(c0+c1+c3) + a1*(c2+c1*(mu + theta)) + a2*c2
q0 = a1*a2*(c2*(c0+c1)*(mu + theta))

In [ ]:
(q1/q3)**2 - q2*q1/q3 + q0